# Aula 04 | Classificação de mensagens e avaliação de LLMs

**Inteligência Artificial para Decisão** · Marketing · notebook local.

**Pergunta:** Quando uma classificação automática de reclamações é suficientemente boa para ser utilizada?

**Procedimento:** execute as células, confira os resultados e modifique uma hipótese. Não há chamadas a serviços externos. Todos os dados são sintéticos ou fictícios.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

# Localiza a raiz do curso, inclusive ao abrir o notebook a partir da pasta da aula.
def achar_raiz():
    origem = Path.cwd().resolve()
    for pasta in (origem, *origem.parents):
        if (pasta / "dados" / "README.md").exists() and (pasta / "src").exists():
            return pasta
    raise FileNotFoundError("Abra o notebook dentro do repositório extraído, preservando dados/ e src/.")

ROOT = achar_raiz()
sys.path.insert(0, str(ROOT / "src"))
print("Repositório:", ROOT)


## Base e classe majoritária

In [ ]:
df=pd.read_csv(ROOT/'dados/avaliacoes_rotuladas.csv')
print(df['categoria'].value_counts().head(9).to_string())
print('Taxa de encaminhamento humano:',df['encaminhar_humano'].mean())

## Matriz de confusão

In [ ]:
from iad.metrics import classification_report_counts
r=classification_report_counts(tp=12,fp=6,fn=8,tn=74)
print({k:round(v,3) for k,v in r.items()})
assert abs(r['precisao']-12/18)<1e-12

## Classificador básico com teste honesto

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
Xtr,Xte,ytr,yte=train_test_split(df['texto'],df['encaminhar_humano'],test_size=.3,random_state=42,stratify=df['encaminhar_humano'])
clf=make_pipeline(TfidfVectorizer(strip_accents='unicode'),LogisticRegression(max_iter=300))
clf.fit(Xtr,ytr)
print(classification_report(yte,clf.predict(Xte),zero_division=0))
print('ATENÇÃO: base sintética com textos repetidos entre treino e teste; NÃO é estimativa de produção.')

## Síntese individual

1. Qual pergunta este experimento responde e qual **não** responde?
2. Que valores foram calculados e quais foram apenas interpretados?
3. Que alteração você realizou? Registre a saída e explique sua escolha.
4. Qual erro seria especialmente relevante para uma decisão em **Marketing**?

**Entrega:** Relatório comparativo de dois prompts, baseline e tabela de erros.